In [0]:
# =============================================================================
# ICEBASE — PHASE 2 | NOTEBOOK 03
# Silver dim_customer — Streaming Table (Python)
# Idaho Mashers Hockey Analytics Platform
# =============================================================================
# PIPELINE SOURCE FILE — DO NOT RUN AS A STANDALONE NOTEBOOK.
# Add to the icebase-bronze-to-silver pipeline as a source file.
#
# LANGUAGE: Python (required — SQL cannot union two independent readStream sources)
#
# TABLE DEFINED HERE:
#   icebase.silver.dim_customer  (Streaming Table)
#
# SOURCES (two streams unioned):
#   1. icebase.bronze.raw_customers       — Phase 1 seed data (5,000 fans)
#      Written as a static Delta table by the seed generator.
#      Read with skipChangeCommits=true because Phase 1 used .mode("append")
#      to add the 412 Jersey Night fans after initial write.
#
#   2. icebase.bronze.raw_customers_stream — Net-new simulator drops
#      Defined in notebook 01. Fully qualified name required because the
#      pipeline default schema is 'silver' — without the full path,
#      Spark would look for icebase.silver.raw_customers_stream and fail.
#
# CLEANING APPLIED IN SILVER:
#   full_name  : INITCAP(TRIM()) — removes dirty whitespace, title-cases ALL CAPS
#   email      : LOWER(TRIM())   — normalizes to lowercase
#   tenure_days: datediff(today, signup_date) — derived fan lifecycle metric
#   record_source: 'seed' or 'simulator' — tracks data lineage per record
#
# QUALITY GATES:
#   expect_or_drop : hard DROP — null customer_id or email removed from Silver
#   expect         : soft WARN — invalid channel flagged but record kept
# =============================================================================

from pyspark import pipelines as dp
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

# -----------------------------------------------------------------------------
# Helper: apply cleaning and column selection to any customer DataFrame.
# Called once for each source so logic stays DRY.
# -----------------------------------------------------------------------------

def _clean_customer(df, source_label: str):
    """Apply Silver-level cleaning and enrichment to a raw customer DataFrame."""
    return df.select(
        F.col("customer_id"),

        # Clean full_name: strip whitespace, title-case (fixes ALL CAPS from seed)
        F.initcap(F.trim(F.col("full_name"))).alias("full_name"),

        # Normalize email to lowercase
        F.lower(F.trim(F.col("email"))).alias("email"),

        F.col("phone"),
        F.col("zip_code"),
        F.col("state"),
        F.col("acquisition_channel"),
        F.col("initial_segment"),
        F.col("is_season_holder"),
        F.col("signup_date"),
        F.col("is_jersey_night_acq"),
        F.col("is_deeply_lapsed"),

        # Fan tenure in days — key feature for segmentation and churn models
        F.datediff(
            F.current_date(),
            F.col("signup_date").cast(DateType())
        ).alias("tenure_days"),

        # Track which system produced this record
        F.lit(source_label).alias("record_source"),

        F.col("_ingested_at"),
    )


# -----------------------------------------------------------------------------
# dim_customer — Streaming Table
# -----------------------------------------------------------------------------

@dp.table(
    comment          = "Silver customer dimension — cleaned, unified from seed + simulator",
    table_properties = {"quality": "silver", "team": "idaho_mashers"}
)
@dp.expect_or_drop("customer_id_not_null", "customer_id IS NOT NULL")
@dp.expect_or_drop("email_not_null",       "email IS NOT NULL")
@dp.expect(
    "valid_acquisition_channel",
    "acquisition_channel IN ("
    "'season_package','walk_up','social_media','friend_referral',"
    "'email_campaign','promo_code','corporate','jersey_night_event')"
)
def dim_customer():

    # ── Source 1: Phase 1 seed customers ─────────────────────────────────
    # skipChangeCommits=true because raw_customers had records appended
    # (Jersey Night fans) after the initial Delta write in Phase 1.
    seed_customers = _clean_customer(
        spark.readStream
            .option("skipChangeCommits", "true")
            .table("icebase.bronze.raw_customers"),
        "seed"
    )

    # ── Source 2: Net-new signups from simulator volume drops ─────────────
    # Fully qualified 3-part name required — pipeline default schema is silver.
    new_customers = _clean_customer(
        spark.readStream
            .option("skipChangeCommits", "true")
            .table("icebase.bronze.raw_customers_stream"),
        "simulator"
    )

    # Union both sources into a single Silver stream
    return seed_customers.union(new_customers)